In [2]:
import pandas as pd
import os
from sqlalchemy import (
    create_engine,
    text,
    Column,
    Integer,
    String,
    Text,
    DateTime,
    UniqueConstraint,
    Float,
    ForeignKey
)
from sqlalchemy.orm import declarative_base, relationship 
from lingua import LanguageDetectorBuilder
from dotenv import load_dotenv
import re
import unicodedata

load_dotenv()

True

In [3]:
# ==========================================================
# CONFIGURATION
# ==========================================================

HOST     = os.getenv("DB_HOST", "localhost")
PORT     = os.getenv("DB_PORT", 5432)
DBNAME   = os.getenv("DB_NAME",   "postgres")
USER     = os.getenv("DB_USER", "postgres")
PASSWORD = os.getenv("DB_PASSWORD", "")

SOURCE_DATABASES = [
    "s1food",
    "safefood",
    "safefood-1",
    "safefood0",
    "safefood1",
    "safefood2",
    "safefooddata",
    "safefooddb",
    "safefoodmod",
    "sfood",
]

TARGET_DATABASE = "safefooddatabase"

TABLE_NAME = "raw_posts"

OUTPUT_EXCEL = "final_raw_posts.xlsx"

SAVE_TO_DATABASE = True

In [4]:
# ==========================================================
# Language Detector
# ==========================================================

_detector = (
    LanguageDetectorBuilder.from_all_languages()
    .with_preloaded_language_models()
    .build()
)


def detect_language(text):
    if not text or not str(text).strip():
        return "unknown"

    try:
        language = _detector.detect_language_of(str(text).strip())

        if language is None:
            return "unknown"

        return str(language.iso_code_639_1).split(".")[-1].lower()

    except Exception:
        return "unknown"

def remove_leading_title(platform,title, full_text):

    if not title or not full_text:
        return full_text
    
    title = str(title).strip()
    full_text = str(full_text).strip()
    if platform.lower().startswith("youtube"):
        if full_text.lower().startswith(title.lower()):

            remaining = full_text[len(title):].lstrip(" :-–—|.,\n\t")

            # Only remove title if something meaningful remains
            if remaining.strip():
                return remaining

    return full_text


def remove_emojis(text):
    """
    Remove emoji characters.
    """

    if not text:
        return text

    return "".join(
        ch
        for ch in text
        if not unicodedata.category(ch).startswith("So")
    )


def trim_to_first_letter(text):
    """
    Remove everything before the first Unicode alphabetic character.
    Keeps letters from any language.
    """

    if not text:
        return text

    for i, ch in enumerate(text):
        if ch.isalpha():
            return text[i:]

    return ""

In [5]:
# ==========================================================
# SQLAlchemy Model
# ==========================================================

Base = declarative_base()

class RawPost(Base):
    __tablename__ = "raw_posts"
    id = Column(Integer, autoincrement=True, primary_key=True)
    platform = Column(String)
    source = Column(String)
    title = Column(String)
    full_text = Column(Text)
    language = Column(String)
    created_date = Column(DateTime(timezone=True))
    week_start_date = Column(DateTime(timezone=True))
    source_url = Column(String, nullable=True)
    post_topic = relationship("PostTopic", back_populates="raw_post", uselist=False)
    __table_args__ = (UniqueConstraint("platform", "source", "title", "created_date"),)


class PostTopic(Base):
    __tablename__ = "posts_with_topics"
    id = Column(Integer, autoincrement=True, primary_key=True)
    raw_post_id = Column(Integer, ForeignKey("raw_posts.id"))
    platform = Column(String)
    source = Column(String)
    title = Column(String)
    full_text = Column(Text)
    week_start_date = Column(DateTime(timezone=True))
    topic_id = Column(Integer)
    topic_label = Column(String)
    food_category = Column(String)
    raw_post = relationship("RawPost", back_populates="post_topic")


class WeeklySnapshot(Base):
    __tablename__ = "weekly_snapshots"
    id = Column(Integer, autoincrement=True, primary_key=True)
    food_category = Column(String)
    topic_label = Column(String)
    week_start_date = Column(DateTime(timezone=True))
    total_posts = Column(Integer)
    platform_count = Column(Integer)
    growth_rate = Column(Float)
    growth_acceleration = Column(Float)
    rolling_avg = Column(Float)
    # Advanced features
    peak_so_far = Column(Float)
    ratio_to_peak = Column(Float)
    weeks_since_peak = Column(Integer)
    trend_3wk = Column(Float)
    sustained_growth = Column(Float)
    posts_lag_1 = Column(Float)
    posts_lag_2 = Column(Float)
    posts_lag_3 = Column(Float)
    # Target
    will_trend = Column(Float)
    __table_args__ = (UniqueConstraint("food_category", "week_start_date"),)


class TrendPrediction(Base):
    __tablename__ = "trend_predictions"
    id = Column(Integer, autoincrement=True, primary_key=True)
    food_category = Column(String)
    topic_label = Column(String)
    # Base features
    total_posts = Column(Integer)
    platform_count = Column(Integer)
    growth_rate = Column(Float)
    growth_acceleration = Column(Float)
    rolling_avg = Column(Float)
    # Advanced features
    peak_so_far = Column(Float)
    ratio_to_peak = Column(Float)
    weeks_since_peak = Column(Integer)
    trend_3wk = Column(Float)
    sustained_growth = Column(Float)
    posts_lag_1 = Column(Float)
    posts_lag_2 = Column(Float)
    posts_lag_3 = Column(Float)
    # Predictions
    trend_probability = Column(Float)
    prediction = Column(Integer)
    prediction_week = Column(DateTime(timezone=True))

In [6]:
# ==========================================================
# Helper
# ==========================================================

def engine(db):
    return create_engine(
        f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{db}",
        future=True,
    )


In [7]:
# ==========================================================
# Read all databases
# ==========================================================

from sqlalchemy import inspect

required_columns = [
    "platform",
    "source",
    "title",
    "full_text",
    "created_date",
]

optional_columns = [
    "source_url",
]

dfs = []

for db in SOURCE_DATABASES:

    print(f"Reading {db}")

    eng = engine(db)

    inspector = inspect(eng)

    # Skip database if table doesn't exist
    if "raw_posts" not in inspector.get_table_names():
        print(f"Skipping {db}: raw_posts table not found")
        continue

    existing_columns = {
        col["name"] for col in inspector.get_columns("raw_posts")
    }

    # Verify required columns
    missing = set(required_columns) - existing_columns

    if missing:
        print(f"Skipping {db}: missing columns {missing}")
        continue

    select_columns = required_columns.copy()

    # Keep source_url if available
    if "source_url" in existing_columns:
        select_columns.append("source_url")

    query = f"""
    SELECT
        {', '.join(select_columns)}
    FROM raw_posts
    """

    df = pd.read_sql(query, eng)

    # Add missing optional columns
    if "source_url" not in df.columns:
        df["source_url"] = None

    dfs.append(df)

merged_df = pd.concat(dfs, ignore_index=True)

Reading s1food
Reading safefood
Reading safefood-1
Reading safefood0
Reading safefood1
Reading safefood2
Reading safefooddata
Reading safefooddb
Reading safefoodmod
Reading sfood


In [8]:
# ==========================================================
# Standardize dates
# ==========================================================

merged_df["created_date"] = pd.to_datetime(
    merged_df["created_date"],
    utc=True,
    errors="coerce",
)

merged_df["created_date"] = (
    merged_df["created_date"]
    .dt.tz_localize(None)
)

# Standardize text fields
for col in ["platform", "source", "title", "full_text"]:
    merged_df[col] = (
        merged_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
    )

# Remove duplicated title from beginning of full_text
merged_df["full_text"] = merged_df.apply(
    lambda row: remove_leading_title(
        row['platform'],
        row["title"],
        row["full_text"],
    ),
    axis=1,
)

# Remove emojis
merged_df["full_text"] = (
    merged_df["full_text"]
    .apply(remove_emojis)
)

# Ensure text starts with first alphabetic character
merged_df["full_text"] = (
    merged_df["full_text"]
    .apply(trim_to_first_letter)
)

# Normalize whitespace
merged_df["full_text"] = (
    merged_df["full_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [9]:
# ==========================================================
# Remove duplicates
# ==========================================================

merged_df = merged_df.drop_duplicates(
    subset=[
        "platform",
        "source",
        "title",
        "full_text",
        "created_date",
    ]
)
merged_df = merged_df.drop_duplicates(
    subset=[
        "full_text",
    ]
)
print("Rows after deduplication:", len(merged_df))

merged_df = merged_df[
    merged_df["title"].notna()
    &
    (merged_df["title"].str.strip() != "")
    &
    merged_df["full_text"].notna()
    &
    (merged_df["full_text"].str.strip() != "")
]

Rows after deduplication: 5992


In [10]:
# ==========================================================
# Calculate language
# ==========================================================

unique_texts = set(merged_df["full_text"].fillna("").unique())


lang_map = {
    text: detect_language(text)
    for text in unique_texts
}

merged_df["language"] = (
    merged_df["full_text"]
    .fillna("")
    .map(lang_map)
)


In [11]:
# ==========================================================
# Week start date (Monday)
# ==========================================================

merged_df["week_start_date"] = (
    merged_df["created_date"]
    - pd.to_timedelta(
        merged_df["created_date"].dt.weekday,
        unit="D",
    )
).dt.normalize()

In [12]:
# ==========================================================
# Missing columns
# ==========================================================

# merged_df["source_url"] = None


In [13]:
# ==========================================================
# Final order
# ==========================================================

merged_df = merged_df[
    [
        "platform",
        "source",
        "title",
        "full_text",
        "language",
        "created_date",
        "week_start_date",
        "source_url",
    ]
]

from datetime import datetime

current_year = datetime.now().year

merged_df = merged_df[
    merged_df["created_date"].notna()
]

merged_df = merged_df[
    merged_df["created_date"].dt.year.isin(
        [current_year, current_year - 1]
    )
]

print(f"Rows after year filter: {len(merged_df):,}")


merged_df = (
    merged_df
    .sort_values("created_date")
    .reset_index(drop=True)
)

# NEW
merged_df = merged_df.drop_duplicates(subset=["platform", "source", "title", "created_date"]).reset_index(drop=True)


Rows after year filter: 5,700


In [14]:
# ==========================================================
# Export Excel
# ==========================================================

merged_df.to_excel(
    OUTPUT_EXCEL,
    index=False,
)

print("Excel exported.")

Excel exported.


In [15]:
# ==========================================================
# Create target database if needed
# ==========================================================

admin_engine = engine("postgres")

with admin_engine.connect() as conn:

    conn.execute(text("COMMIT"))

    exists = conn.execute(
        text(
            """
            SELECT 1
            FROM pg_database
            WHERE datname=:db
            """
        ),
        {"db": TARGET_DATABASE},
    ).scalar()

    if not exists:
        print("Creating database...")
        conn.execute(text(f'CREATE DATABASE "{TARGET_DATABASE}"'))

In [16]:

# ==========================================================
# Create table
# ==========================================================

target_engine = engine(TARGET_DATABASE)

Base.metadata.create_all(target_engine)

In [17]:
# ==========================================================
# Bulk insert
# ==========================================================
if SAVE_TO_DATABASE:
        try:
            print("Inserting into database...")

            merged_df.to_sql(
                TABLE_NAME,
                target_engine,
                if_exists="append",
                index=False,
                chunksize=1000,
            )

            print("Insert completed.")

        except Exception as e:
            import traceback
            traceback.print_exc()
else:
    print(f"end=of-script: SAVE_TO_DATABASE is set to False. No data was inserted into the database.")

Inserting into database...
Insert completed.


In [18]:
import pandas as pd
OUTPUT_EXCEL = "final_raw_posts.xlsx"
data=pd.read_excel(
    OUTPUT_EXCEL,
    engine="openpyxl",
    dtype={
        "platform": str,
        "source": str,
        "title": str,
        "full_text": str,
        "language": str,
        "created_date": "datetime64[ns]",
        "week_start_date": "datetime64[ns]",
        "source_url": str,
    },
)
# keep=False marks ALL instances of a duplicate as True (not just the 2nd, 3rd, etc.)
duplicates = data[data['full_text'].duplicated(keep=False)]

# Group by the text to see how many times each appears
duplicate_summary = duplicates.groupby('full_text').size().reset_index(name='count')

print(duplicate_summary.sort_values(by='count', ascending=False))


Empty DataFrame
Columns: [full_text, count]
Index: []
